In [4]:
!nvidia-smi

Thu Jun 26 16:29:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install transformers==4.38.0
!pip install peft==0.10.0
!pip install accelerate==0.28.0

In [6]:
import os
import shutil
import random

import pandas as pd
import numpy as np
import torch
from torch.nn.functional import one_hot
from torch import LongTensor, FloatTensor
from transformers import AutoTokenizer, DataCollatorWithPadding, pipeline, \
    AutoModelForSequenceClassification as AMFSC, TrainingArguments, Trainer
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from datasets import Dataset, DatasetDict

In [8]:
train = pd.read_csv('train.csv', index_col=0)
test = pd.read_csv('test.csv', index_col=0)

In [9]:
df = pd.concat([train, test])

In [10]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
SEED = 111
seed_everything(SEED)

In [11]:
id2label = {0: "negative", 1: "neutral", 2: "positive", 3: "garbage"}
label2id = {value: key for key, value in id2label.items()}
num_labels = 4

In [12]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=111)

In [13]:
y_train = y_train.values
y_test = y_test.values

y_train = y_train.astype(int)
y_test = y_test.astype(int)

X_test = X_test.astype(str)
X_train = X_train.astype(str)

In [14]:
train_one_hot = one_hot(LongTensor(y_train))
train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": train_one_hot.type(FloatTensor)
})
train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 23244
})

In [15]:
test_one_hot = one_hot(LongTensor(y_test))
test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": test_one_hot.type(FloatTensor)
})

In [16]:
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 23244
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 5812
    })
})

In [17]:
model_path = "ai-forever/ruRoberta-large"

In [18]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000
)
tokenized_dataset = tokenized_dataset.remove_columns("text")
tokenized_dataset["test"][0]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/23244 [00:00<?, ? examples/s]

Map:   0%|          | 0/5812 [00:00<?, ? examples/s]

{'label': [1.0, 0.0, 0.0, 0.0],
 'input_ids': [1, 7865, 2653, 26165, 225, 2],
 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [19]:
model = AMFSC.from_pretrained(
    model_path,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ai-forever/ruRoberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

count_parameters(model) / 1_000_000

426.912772

In [21]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    labels = np.argmax(labels, axis=1)
    results = {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),
        "precision": precision_score(
            labels,
            predictions,
            average='weighted',
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            average='weighted',
            zero_division=0
        ),
        "f1_score": f1_score(
            labels,
            predictions,
            average='weighted',
            zero_division=0
        )
    }
    return results

In [22]:
folder_path = '-'.join(model_path.split('/'))
prefix = './ckpt'

In [23]:
best_trial = [
    0,
    0,
    {
        'per_device_train_batch_size': 4,
        'per_device_eval_batch_size': 4,
        'num_train_epochs': 20,
        'learning_rate': 1.8805677788552795e-05,
        'weight_decay': 0.009912281337200485
    }
]

In [24]:
from transformers import TrainingArguments

In [25]:
training_args = TrainingArguments(
    output_dir=f"{prefix}/best_models/{folder_path}",
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    **best_trial[2]
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Score
1,0.213500,0.202555,0.875602,0.883346,0.875602,0.878131
2,0.184900,0.188469,0.893840,0.890834,0.893840,0.891316
3,0.168500,0.217269,0.899174,0.898876,0.899174,0.898367
4,0.132500,0.246331,0.893668,0.894730,0.893668,0.893060
5,0.101300,0.218306,0.892464,0.901330,0.892464,0.895199
6,0.078900,0.221743,0.913111,0.911357,0.913111,0.910832


KeyboardInterrupt: 

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
model_path = 'Aniemore/rubert-large-emotion-russian-cedr-m7'

In [28]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000
)
tokenized_dataset = tokenized_dataset.remove_columns("text")
tokenized_dataset["test"][0]

tokenizer_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/23244 [00:00<?, ? examples/s]

Map:   0%|          | 0/5812 [00:00<?, ? examples/s]

{'label': [1.0, 0.0, 0.0, 0.0],
 'input_ids': [101, 16377, 18972, 102],
 'token_type_ids': [0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1]}

In [29]:
model = AMFSC.from_pretrained(
    model_path,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at Aniemore/rubert-large-emotion-russian-cedr-m7 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([7]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([7, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
count_parameters(model) / 1_000_000

426.912772

In [33]:
training_args = TrainingArguments(
    output_dir=f"{prefix}/best_models/{folder_path}",
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    **best_trial[2]
)

In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Score
1,0.172900,0.166485,0.897970,0.899150,0.897970,0.897955
2,0.138400,0.174282,0.902615,0.904100,0.902615,0.902588
3,0.109500,0.212972,0.909153,0.909566,0.909153,0.909207
4,0.071000,0.243330,0.906056,0.907148,0.906056,0.906424
5,0.052700,0.237167,0.906573,0.905947,0.906573,0.905847


KeyboardInterrupt: 